# 01 · ICNALE GRA — build the pool

*Holistic essay score band (Low / Mid / High)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** Asian-learner L2 English essays, each rated on holistic and analytic scales by many trained raters. This is an **automated writing evaluation** task: whole essays, not sentences.

**Difficulty of the labeling judgment:** ★★☆ — moderate, but a different shape of task: long texts and an ordered scale.

**Licence:** ⚠️ **Research use only — NOT redistributable.** Requires registration. Nothing derived from it may be committed to git or included in your submission bundle.  
**Cite:** Ishikawa, S. *The ICNALE Global Rating Archives.*

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

Those three keys are required on every track. Two tracks add more: `cars50` and `raamove` ask what a sentence *does in a passage*, which is not always decidable from the sentence on its own, so their items also carry `doc_id`, `sent_index`, `n_sents` and `context`. Extra keys are safe everywhere — nothing in the pipeline checks for keys it does not need.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work in the RUNTIME, not in Drive: the next cells download a whole
    # corpus, and raw data is big, mostly not ours to redistribute, and one
    # command to fetch again. The pool you build from it is what persists.
    os.makedirs("/content/raw", exist_ok=True)
    os.chdir("/content/raw")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH,
                    DEV_PATH, TEST_PATH, DISAGREED_PATH, PRED_PATH,
                    ROUNDS_PATH, TESTLOG_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)
from pathlib import Path

describe()                  # what this notebook is working on


## Step 1 — Get the data (this one is manual)

ICNALE GRA is released for research use behind a registration form that emails you a password. There is nothing to automate, and that is deliberate — the licence does not permit redistribution.

1. Register at <https://language.sakura.ne.jp/icnale/download.html> and wait for the password.
2. Download and unpack `ICNALE_GRA_2.x.zip`.
3. From its rating tables, export a CSV with **exactly two columns**, `text` and `score`.

In Colab, the cell below opens a file picker. Every other track downloads its corpus in one command and so keeps the raw data in the runtime — this one you cannot re-fetch without going back through the registration form, so it is worth keeping the file in your group's Drive folder and uploading it only once. The second option below does that.

⚠️ `data/raw/` is excluded from git and from your submission bundle, and anything with `icnale` in the name is excluded twice over. Leave it that way — the licence does not permit redistribution.

**First, get the file into this session.** Run the cell below and a file picker opens; choose your `essays_scores.csv`.

Skip this cell if you have already put the file in your group's Drive folder — the next cell points at it there.

In [ ]:
from google.colab import files

files.upload()

**Now we say where the file is.** One of the two lines below is live and the other is commented out. Keep the first if you just uploaded the file; switch to the second, by moving the `#`, once you have put a copy in your group's Drive folder and want to stop uploading it every session.

In [ ]:
RAW_FILE = "essays_scores.csv"        # the copy you just uploaded

# The copy in your group's Drive folder, if you put one there:
# RAW_FILE = str(ROOT / "data" / "raw" / "icnale" / "essays_scores.csv")

print("using", RAW_FILE)

**Now we check the file is really there.** We say so here rather than three cells down, where the same problem arrives as a bare `FileNotFoundError` from inside `open`, with nothing to tell you what to do about it.

In [ ]:
import os

if not os.path.isfile(RAW_FILE):
    raise FileNotFoundError(
        RAW_FILE + " is not here.\n"
        "This is the one track with no automatic download: register at "
        "https://language.sakura.ne.jp/icnale/download.html, export a CSV "
        "with a text column and a score column, then either uncomment the "
        "upload line above and run this cell again, or put the file in "
        "data/raw/icnale/ in your group's Drive folder and use the second "
        "RAW_FILE line.")

print("found it:", RAW_FILE)

## Step 2 — Look at the raw format

The cell prints the **distribution** of the scores, not just a couple of rows. You need that before step 3: it is what tells you where cutting the scale leaves you with three usable classes rather than one big one and two nearly empty ones.

In [ ]:
import csv

# Collect the scores. They arrive as strings, so each one is converted to a
# number - and a cell that is not a number at all is counted, not ignored.
scores = []
not_a_number = 0
with open(RAW_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("columns:", reader.fieldnames)
    for row in reader:
        try:
            scores.append(float(row["score"]))
        except (TypeError, ValueError):
            not_a_number = not_a_number + 1

print(len(scores), "usable scores ·", not_a_number, "cells that were not numbers")

Now we look at how those scores are spread out.

Read the percentiles carefully — they are what step 3 asks you to cut the scale with. Cutting at the 33rd and 67th gives you three classes of roughly equal size; cutting anywhere else does not, and you will see that in step 4.

In [ ]:
scores.sort()             # smallest first, so we can pick positions out of it
print("min", scores[0], "· max", scores[-1])

for q in (10, 25, 33, 50, 67, 75, 90):
    # A rough percentile: the score q% of the way along the sorted list.
    position = int(len(scores) * q / 100)
    print("   ", str(q) + "th percentile:", scores[position])

### Reading CSV with `csv.DictReader`

`csv.DictReader` reads a CSV using its header row, so each row arrives as a dict keyed by column name — `row["score"]` rather than `row[1]`. That is what the cell above used to collect the scores.

**The two columns that matter:**

* `text` — the essay
* `score` — the holistic score. **It arrives as a string**, even when it looks like a number, so it has to be converted before it can be compared to a boundary.

The reshaping function below uses exactly these:

1. `open(..., encoding="utf-8-sig")` — strips the byte-order mark this file ships with, which would otherwise be glued to the first column name.
2. `csv.DictReader(handle)` — iterate rows as `{column: value}` dicts.
3. `float(raw_score)` inside a `try` — a non-numeric cell is counted and skipped rather than crashing the run. The function prints how many it skipped; if that number is not small, look at the file before trusting the rest.
4. `score < low_below` / `score < mid_below` — the two cut-offs you are about to choose. Everything else in the function is fixed; this is the whole decision.

## Step 3 — Reshape into the canonical schema

One decision, and it is entirely yours: ✏️ **where do the band boundaries go?**

There is no right answer sitting in the data waiting to be found. Two honest ways to choose, and they disagree:

- **From the rubric** — if the scale you are using says what a Low essay is, use that. Your classes will come out uneven, possibly badly, but they mean something outside your own study.
- **From the distribution** — cut at the 33rd and 67th percentiles (printed above) and your classes come out balanced. Your F1 is then easier to read, and your bands mean nothing except "bottom third of this sample".

Pick one, say which in `PLAN.md`, and report the boundaries as numbers. A band definition that exists only as an unexplained `4.0` in a notebook is not a scheme.

⚠️ These labels are **ordered** (Low < Mid < High) but they are *not* alphabetical. List them under `labels_order:` in `config.yaml` — Low, then Mid, then High — or the weighted κ gets computed over `High < Low < Mid`, which means nothing.

### The code that does it — read it, then run it

Three functions. `reshape_icnale` is the one to read: the two boundaries you are about to choose are its only arguments.

It is read straight out of `scripts/reshape.py` when this notebook is generated, so it is not a simplified copy: it is the code that runs.

It arrives one function per cell, so you can take them one at a time. **None of these cells print anything.** They only give the functions their names — that is what `def` does. You will see no output until the cell *after* them, which calls one.

First, the two library modules the code below needs. `import` is how Python is told to load one.

In [ ]:
import csv

`reid` renumbers items 1, 2, 3 … so that every item has an id of its own.

In [ ]:
def reid(items: list[dict[str, str]]) -> list[dict[str, str]]:
    """Renumber ids sequentially from 1, keeping the current order.

    Args:
        items: the items to renumber. They are copied, not changed in place.

    Returns:
        The same items with new ids.
    """
    renumbered = []
    next_id = 1

    for item in items:
        ### Copy before writing ###
        new_item = dict(item)                    # Work on a copy, so the caller's item is left alone.

        ### Stamp the id ###
        new_item["id"] = next_id                 # Overwrite whatever id was there with the running number.
        renumbered.append(new_item)              # Keep it in the order it arrived.
        next_id = next_id + 1                    # Advance, so the next item gets a fresh id.

    return renumbered

`reshape_icnale` reads the CSV and turns each score into a band. Find the `if / elif / else` — those three lines are the whole scheme.

In [ ]:
def reshape_icnale(csv_path: str | Path, low_below: float = 4.0,
                   mid_below: float = 7.0) -> list[dict[str, str]]:
    """Band a numeric holistic score into Low / Mid / High.

    Args:
        csv_path: the downloaded CSV, with a text column and a score column.
        low_below: scores under this are Low.
        mid_below: scores under this, and not Low, are Mid. The rest are High.

    Returns:
        The items in canonical form.

    Note:
        THE CUT-OFFS ARE PLACEHOLDERS. 4 and 7 are not from the ICNALE rubric - they
        are round numbers. Where you put the boundaries decides how hard the task is
        and how balanced the classes are, so set them from the rubric you are
        actually using and say what you chose in your report.
    """
    rows = []
    skipped = 0
    with open(csv_path, encoding="utf-8-sig", newline="") as handle:

        ### Read the CSV a row at a time ###
        for record in csv.DictReader(handle):    # DictReader gives each row as {column: value}.

            ### Pull the essay and its score ###
            text = (record.get("text") or "").strip()
            raw_score = (record.get("score") or "").strip()   # Still a string at this point.
            if not text or not raw_score:        # Nothing to band -> skip.
                continue

            ### Turn the score into a number ###
            try:
                score = float(raw_score)
            except ValueError:
                skipped = skipped + 1      # a non-numeric cell: report it, do not crash
                continue

            ### Apply the two cut-offs ###
            if score < low_below:                # Everything below the first boundary.
                label = "Low"
            elif score < mid_below:              # Between the two boundaries.
                label = "Mid"
            else:                                # At or above the second boundary.
                label = "High"
            rows.append({"id": 0, "text": text, "label": label})
    if skipped:
        print("  note: skipped", skipped, "row(s) whose score cell was not a number.")
    return reid(rows)

`validate` checks that every item has an id, a text and a label. Nothing calls it here — step 5 does, just before saving.

In [ ]:
def validate(items: list[dict[str, str]],
             allowed: list[str] | None = None) -> None:
    """Check the canonical schema, and raise on the first problem found.

    Deliberately explicit rather than `assert`: assertions vanish under `python -O`,
    and a silently unvalidated dataset is exactly the kind of thing that surfaces as a
    baffling metric three days later.

    Args:
        items: the items to check, each needing "id", "text" and "label".
        allowed: the labels the scheme allows. Left out, any label passes.

    Returns:
        Nothing.

    Raises:
        ValueError: on the first item that is missing a field, has a repeated id, or
            carries a label outside `allowed`.
    """
    seen_ids = set()
    for position, item in enumerate(items):
        where = "Item number " + str(position + 1) + " of " + str(len(items))
        for field in ("id", "text", "label"):
            if field not in item:
                raise ValueError(
                    where + " has no `" + field + "`, and every item needs all three of "
                    "id, text and label.\n"
                    "That item was built by the reshaping step above, so go back to the "
                    "cell where you filled in your own decision and check it puts a "
                    "`" + field + "` on every row.")
        if item["id"] in seen_ids:
            raise ValueError(
                "Two items have the same id (" + str(item["id"]) + "), so one would "
                "overwrite the other in your annotation sheet.\n"
                "reid() renumbers everything 1, 2, 3 - make sure the last line of your "
                "reshaping step hands its rows to it.")
        seen_ids.add(item["id"])
        if not isinstance(item["text"], str) or not item["text"].strip():
            raise ValueError(
                "The item with id " + str(item["id"]) + " has no text - there is nothing "
                "there for a coder or the model to read.\n"
                "Blank rows usually come from the raw file. Skip them in the reshaping "
                "step rather than annotate them.")
        if not isinstance(item["label"], str) or not item["label"]:
            raise ValueError(
                "The item with id " + str(item["id"]) + " has no label.\n"
                "Every item in a pool needs the published label it came with. If your "
                "label mapping does not cover some code in the raw data, either add it "
                "or drop those rows in the reshaping step - do not leave the label blank.")
        if allowed is not None and item["label"] not in allowed:
            raise ValueError(
                "The item with id " + str(item["id"]) + " has the label '"
                + str(item["label"]) + "', which is not one of the labels you allowed:\n"
                "  " + ", ".join(sorted(allowed)) + "\n"
                "Either add it to your label set, or map it onto one of these in the "
                "cell where you wrote your label mapping.")

## Step 3a — Cut the scale

Now we turn each numeric score into one of three bands. The two boundaries are yours to choose, and the percentiles printed above are the evidence for choosing them.

The defaults, 4.0 and 7.0, are round numbers rather than a rubric — leaving them is as much a decision as changing them, and you have to defend it either way. Run it a couple of ways and look at step 4 each time: watching the counts move as you shift a boundary is the point.

**Whatever you settle on goes in `PLAN.md`** as two numbers and a reason. Do not re-cut the scale after you have seen your F1.

In [ ]:
# ✏️ Step 3a · Cut the scale ─────────────────────────────────────
# Turns each numeric score into a Low, Mid or High band, split at the two
# boundaries you give it.
# Creates: rows (a list)

# ✏️ this runs as written — the work is deciding whether it should

# A score below low_below is Low; below mid_below is Mid; the rest High.
#
# 4.0 and 7.0 are the defaults, and they are round numbers rather than
# a rubric — leaving them is as much a decision as changing them, and
# you have to defend it either way. Re-run with the percentiles printed
# above and watch the counts in step 4 move.
rows = reshape_icnale(RAW_FILE, low_below=4.0, mid_below=7.0)

print(len(rows), "essays")


## Step 4 — Check the label balance

Now we look at what your two boundaries actually produced. If one band has almost everything in it, go back to step 3a and move a cut-off.

Now we count what we have got: how many items, how many of each label, and which fields every item carries.

In [ ]:
# Count how many items carry each label, one item at a time.
label_counts = {}
for item in rows:
    label = item["label"]
    if label not in label_counts:
        label_counts[label] = 0
    label_counts[label] = label_counts[label] + 1

print("total items:", len(rows))
print("label counts:", label_counts)
print("fields per item:", list(rows[0].keys()))

Now we look at three whole items, to see the shape of one.

Where a track carries a `context` (the whole passage a sentence came from), it is shortened here so it does not bury everything else. That only changes what is **printed** — `rows` itself is untouched.

In [ ]:
for item in rows[:3]:
    preview = dict(item)          # a copy, so trimming it changes nothing
    if preview.get("context"):
        preview["context"] = preview["context"][:70] + " …"
    print(preview)
    print("---")

## Step 5 — Save it

Three short cells: check that `config.yaml` agrees which track this is, check the shape of every item, then write the file.

⚠️ Keep this file **out of git** and **out of your submission bundle**. `.gitignore` and `scripts/make_submission.py` both exclude anything with `icnale` in the name — please leave that in place.

**First, a safety check.** `POOL_PATH` is built from the `track:` line in `config.yaml`. If that still says another track, saving now would write icnale data into a file belonging to something else — and everything downstream would run perfectly on the wrong data. The first sign of trouble would be labels that make no sense in notebook 03, by which point two people have annotated forty items.

If this cell stops you: open `config.yaml`, set `track:` to `icnale`, save it, then re-run the SETUP cell at the top of this notebook.

In [ ]:
if TRACK not in ['icnale']:
    raise RuntimeError(
        "config.yaml says  track: " + str(TRACK) + "  but this is the icnale "
        "notebook, so saving now would put icnale data into "
        + POOL_PATH.name + ", which belongs to another track.\n"
        "Open config.yaml, set  track: to one of icnale,"
        " save it, then re-run the SETUP cell at the top of this notebook.")

print("config.yaml agrees: this is the", TRACK, "track.")

**Now we check the shape of every item.** Everything downstream — the sampling, the annotation sheet, the scoring — assumes each item has an `id`, a `text` and a `label`. A pool that breaks that assumption does not fail here; it fails in notebook 03, after two people have annotated forty items.

`validate` says nothing when all is well. Silence is the pass.

In [ ]:
validate(rows)
print("All", len(rows), "items have an id, a text and a label.")

**Now we write the pool** into your group's Drive folder, under the exact name notebook 02 will look for. Both notebooks get that name from `config.yaml`, so there is nothing to copy or paste between them.

In [ ]:
import json

POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(POOL_PATH, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", POOL_PATH)

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** open `02_sample.ipynb`. It reads `POOL_PATH` — the file the cell above just wrote, in your group's Drive folder. Nothing to copy, nothing to paste: that path is the handoff, and both notebooks get it from the same `config.yaml`.